In [1]:
# ============================================================
# Count max / average Qwen tokenizer tokens for:
# 1) relations[*]["relation"]
# 2) facts[*]["info"]
# in a JSON file on Google Drive
# ============================================================

!pip -q install transformers pandas tqdm

from google.colab import drive
drive.mount("/content/drive")

import json
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer

Mounted at /content/drive


In [2]:
# -----------------------------
# Config
# -----------------------------
MODEL_NAME = "Qwen/Qwen3-Embedding-8B"

JSON_PATH = "/content/drive/MyDrive/final_project/idea_1/kg/hotpotqa/hotpotqa_kg_extractions_all_00000001_to_00035029.clean.json"

ADD_SPECIAL_TOKENS = True

BATCH_SIZE = 1024

THRESHOLDS = [128, 256, 512, 1024]


# -----------------------------
# Load tokenizer only, not model
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    trust_remote_code=True,
)

print("Loaded tokenizer:", MODEL_NAME)
print("Tokenizer model max length:", tokenizer.model_max_length)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/729 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

Loaded tokenizer: Qwen/Qwen3-Embedding-8B
Tokenizer model max length: 131072


In [3]:
# -----------------------------
# Load JSON
# -----------------------------
with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

print("Number of chunks:", len(data))


# -----------------------------
# Extract texts
# -----------------------------
relation_rows = []
fact_info_rows = []

for obj_i, item in enumerate(tqdm(data, desc="Extracting relation/info texts")):
    chunk_id = item.get("chunk_id")
    title = item.get("title")
    input_index = item.get("input_index")

    for rel_i, rel in enumerate(item.get("relations") or []):
        text = rel.get("relation")
        if text is not None:
            relation_rows.append({
                "section": "relations.relation",
                "chunk_id": chunk_id,
                "title": title,
                "input_index": input_index,
                "local_index": rel_i,
                "text": str(text),
                "head": rel.get("head"),
                "tail": rel.get("tail"),
            })

    for fact_i, fact in enumerate(item.get("facts") or []):
        text = fact.get("info")
        if text is not None:
            fact_info_rows.append({
                "section": "facts.info",
                "chunk_id": chunk_id,
                "title": title,
                "input_index": input_index,
                "local_index": fact_i,
                "text": str(text),
                "entity": fact.get("entity"),
            })

print("Number of relation texts:", len(relation_rows))
print("Number of fact info texts:", len(fact_info_rows))


# -----------------------------
# Token counting function
# -----------------------------
def count_tokens_for_rows(rows, batch_size=1024):
    counts = []

    for start in tqdm(range(0, len(rows), batch_size), desc="Counting tokens"):
        batch_rows = rows[start:start + batch_size]
        texts = [r["text"] for r in batch_rows]

        encoded = tokenizer(
            texts,
            add_special_tokens=ADD_SPECIAL_TOKENS,
            padding=False,
            truncation=False,
            return_attention_mask=False,
        )

        batch_counts = [len(ids) for ids in encoded["input_ids"]]
        counts.extend(batch_counts)

    for row, count in zip(rows, counts):
        row["token_count_qwen"] = count

    return rows


relation_rows = count_tokens_for_rows(relation_rows, BATCH_SIZE)
fact_info_rows = count_tokens_for_rows(fact_info_rows, BATCH_SIZE)


# -----------------------------
# Convert to DataFrames
# -----------------------------
df_rel = pd.DataFrame(relation_rows)
df_fact = pd.DataFrame(fact_info_rows)

df_all = pd.concat([df_rel, df_fact], ignore_index=True)


# -----------------------------
# Summary stats
# -----------------------------
def make_summary(df, section_name):
    s = df["token_count_qwen"]

    row = {
        "section": section_name,
        "count": int(s.count()),
        "min": int(s.min()) if len(s) else None,
        "max": int(s.max()) if len(s) else None,
        "mean": float(s.mean()) if len(s) else None,
        "median": float(s.median()) if len(s) else None,
        "p90": float(s.quantile(0.90)) if len(s) else None,
        "p95": float(s.quantile(0.95)) if len(s) else None,
        "p99": float(s.quantile(0.99)) if len(s) else None,
    }

    for t in THRESHOLDS:
        row[f"count_>{t}"] = int((s > t).sum())
        row[f"percent_>{t}"] = float((s > t).mean() * 100) if len(s) else None

    return row


summary_df = pd.DataFrame([
    make_summary(df_rel, "relations.relation"),
    make_summary(df_fact, "facts.info"),
])

print("\n========== SUMMARY ==========")
display(summary_df)


# -----------------------------
# Top longest examples
# -----------------------------
TOP_K = 20

top_rel = (
    df_rel
    .sort_values("token_count_qwen", ascending=False)
    .head(TOP_K)
    [["token_count_qwen", "chunk_id", "title", "input_index", "local_index", "text", "head", "tail"]]
)

top_fact = (
    df_fact
    .sort_values("token_count_qwen", ascending=False)
    .head(TOP_K)
    [["token_count_qwen", "chunk_id", "title", "input_index", "local_index", "text", "entity"]]
)

print("\n========== TOP LONGEST relations.relation ==========")
display(top_rel)

print("\n========== TOP LONGEST facts.info ==========")
display(top_fact)


Number of chunks: 35029


Extracting relation/info texts:   0%|          | 0/35029 [00:00<?, ?it/s]

Number of relation texts: 692402
Number of fact info texts: 559959


Counting tokens:   0%|          | 0/677 [00:00<?, ?it/s]

Counting tokens:   0%|          | 0/547 [00:00<?, ?it/s]

...

,section,count,min,max,mean,median,p90,p95,p99,count_>128,percent_>128,count_>256,percent_>256,count_>512,percent_>512,count_>1024,percent_>1024
0,relations.relation,692402,2,100,16.336097,16.0,23.0,25.0,30.0,0,0.0,0,0.0,0,0.0,0,0.0
1,facts.info,559959,2,113,21.570935,21.0,29.0,32.0,38.0,0,0.0,0,0.0,0,0.0,0,0.0



========== TOP LONGEST relations.relation ==========


,token_count_qwen,chunk_id,title,input_index,local_index,text,head,tail
562591,100,hotpotqa_chunk_00028392,Test Icicles,28391,26,Dev and Sam Reunited to play a DJ set at New Y...,Dev,Sam Reunited to play a DJ set at New York's Gl...
23268,76,hotpotqa_chunk_00001132,Ephigenia of Ethiopia,1131,24,"Os dous atlantes da Ethiopia Santo Elesbaõ, em...","Os dous atlantes da Ethiopia Santo Elesbaõ, em...",1735-38
23266,72,hotpotqa_chunk_00001132,Ephigenia of Ethiopia,1131,22,José Pereira de Santana wrote Os dous atlantes...,José Pereira de Santana,"Os dous atlantes da Ethiopia Santo Elesbaõ, em..."
23267,70,hotpotqa_chunk_00001132,Ephigenia of Ethiopia,1131,23,"Os dous atlantes da Ethiopia Santo Elesbaõ, em...","Os dous atlantes da Ethiopia Santo Elesbaõ, em...",Lisboa
441101,66,hotpotqa_chunk_00022273,St Dionis Backchurch,22272,13,St Dionis Backchurch is part of the combined p...,St Dionis Backchurch,"St Edmund the King and Martyr, and St Mary Woo..."
487003,65,hotpotqa_chunk_00024565,Steven Stalinsky,24564,0,Steven Stalinsky wrote the research paper 'Fro...,Steven Stalinsky,"From Al-Qaeda To The Islamic State (ISIS), Jih..."
334257,62,hotpotqa_chunk_00016825,Adoption of Children Act 1949,16824,10,The Adoption of Children Act 1949 may be cited...,Adoption of Children Act 1949,"Adoption of Children (Scotland) Acts, 1930 to,..."
44864,59,hotpotqa_chunk_00002205,122nd Division (Imperial Japanese Army),2204,10,The 122nd Division was part of a batch compris...,122nd Division,128th Division
44863,59,hotpotqa_chunk_00002205,122nd Division (Imperial Japanese Army),2204,9,The 122nd Division was part of a batch compris...,122nd Division,127th Division
44860,59,hotpotqa_chunk_00002205,122nd Division (Imperial Japanese Army),2204,6,The 122nd Division was part of a batch compris...,122nd Division,124th Division



========== TOP LONGEST facts.info ==========


,token_count_qwen,chunk_id,title,input_index,local_index,text,entity
64383,113,hotpotqa_chunk_00003985,John Amos,3984,2,John Amos's film appearances include Vanishing...,John Amos
475038,99,hotpotqa_chunk_00029628,Palm Springs International Film Festival,29627,0,The Palm Springs International Film Festival a...,Palm Springs International Film Festival
226126,91,hotpotqa_chunk_00014031,"Keith County, Nebraska",14030,0,"In Keith County, Nebraska, 25.30% of the popul...","Keith County, Nebraska"
166940,90,hotpotqa_chunk_00010318,"Casey County, Kentucky",10317,3,"The racial makeup of Casey County, Kentucky as...","Casey County, Kentucky"
394211,87,hotpotqa_chunk_00024527,"Chouteau County, Montana",24526,0,"In Chouteau County, Montana, the population di...","Chouteau County, Montana"
61466,86,hotpotqa_chunk_00003792,"San Luis Obispo, California",3791,12,"The population of San Luis Obispo, California ...","San Luis Obispo, California"
509989,86,hotpotqa_chunk_00031847,Symphony No. 2 (Rachmaninoff),31846,3,Symphony No. 2 (Rachmaninoff) is scored for fu...,Symphony No. 2 (Rachmaninoff)
454519,82,hotpotqa_chunk_00028312,Battle of Berestechko,28311,14,"On 19 June 1651, the Polish army numbered 14,8...",19 June 1651
60066,82,hotpotqa_chunk_00003702,"Oakland, California",3701,5,"The population of Oakland, California included...","Oakland, California"
405639,82,hotpotqa_chunk_00025229,Rajesh Roshan,25228,6,Rajesh Roshan composed music for films includi...,Rajesh Roshan
